# 📊 ShopSense — SQL Analysis Notebook
## Phase 2: Database Setup & Analytical Queries

**Objective:** Establish MySQL database infrastructure and execute multi-stage SQL queries to extract cohort retention, funnel metrics, RFM segmentation, and profit insights from raw e-commerce data.

### Analysis Pipeline
| Component | Purpose | Key Metric |
|-----------|---------|-----------|
| **Database Loading** | Import 8 CSV datasets into MySQL | [VALUE] total records |
| **Cohort Analysis** | Track customer retention by acquisition month | [VALUE]% month-1 retention |
| **Funnel Analysis** | Category-level performance metrics | [VALUE] categories analyzed |
| **RFM Segmentation** | Customer value classification | [VALUE] VIP customers |
| **Profit Queries** | Revenue & margin analysis with cost model | [VALUE]% avg margin |

### Key Assumptions
- **Cost Model**: Shipping 12%, Discount 8%, Operations R$5/order
- **Data Filtering**: Excluded cancelled and unavailable orders
- **Cohort Period**: Monthly cohorts based on first purchase date

In [ ]:
from sqlalchemy import create_engine
import pandas as pd

# ──────────────────────────────────────────────────────────────────
# STEP 1: DATABASE SETUP & CONNECTION
# ──────────────────────────────────────────────────────────────────
# Initialize MySQL connection using SQLAlchemy for secure credential handling
# Database: shopsense | Host: localhost (127.0.0.1) | Port: 3306 (default)

engine = create_engine(
    "mysql+mysqlconnector://",
    connect_args={
        "host": "127.0.0.1",
        "user": "root",
        "password": "Shifa@0828",  
        "database": "shopsense"
    }
)

# Verify connection before proceeding with data load
with engine.connect() as conn:
    print("✅ Connected successfully!")

# ──────────────────────────────────────────────────────────────────
# STEP 2: LOAD RAW DATASETS INTO MYSQL
# ──────────────────────────────────────────────────────────────────
# Load 8 datasets from CSV files: customers, orders, products, sellers, 
# payments, reviews, geolocation, and category translations
# Using 'replace' mode ensures fresh data load on each run

files = {
    'orders': '../data/raw/olist_orders_dataset.csv',
    'order_items': '../data/raw/olist_order_items_dataset.csv',
    'customers': '../data/raw/olist_customers_dataset.csv',
    'products': '../data/raw/olist_products_dataset.csv',
    'sellers': '../data/raw/olist_sellers_dataset.csv',
    'payments': '../data/raw/olist_order_payments_dataset.csv',
    'reviews': '../data/raw/olist_order_reviews_dataset.csv',
    'product_category_name_translation': '../data/raw/product_category_name_translation.csv'
}

for table_name, filepath in files.items():
    df = pd.read_csv(filepath)
    df.to_sql(table_name, engine, if_exists='replace', index=False)
    print(f'Loaded {table_name}: {len(df):,} rows')

---

## 📈 Cohort Retention Analysis

**What is it?** Cohort analysis segments customers by their acquisition month (cohort), then tracks what percentage returns in subsequent months. This reveals customer lifetime value and engagement patterns.

**Business Impact:**
- Identifies which acquisition periods yield loyal customers
- Compares retention trends across different time periods
- Indicates if product improvements increase loyalty
- Guides customer acquisition strategy ROI

**Metrics Explained:**
- `cohort_size`: Customers acquired in that month
- `month_1` through `month_6`: Count of cohort members who made repeat purchases 1-6 months later
- **Retention Rate (%)** = (Month N count / Cohort Size) × 100

**Key Finding:** [VALUE]% of customers make a repeat purchase within 1 month of their first order

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# ──────────────────────────────────────────────────────────────────
# STEP 3: COHORT RETENTION ANALYSIS (MySQL CTE Approach)
# ──────────────────────────────────────────────────────────────────
# Multi-stage CTE pipeline:
# 1. customer_cohort: Find each customer's acquisition month
# 2. customer_orders: All orders with customer IDs and purchase month
# 3. cohort_data: Join to calculate months since first purchase
# 4. Final SELECT: Aggregate retention per cohort month

url_object = URL.create(
    "mysql+mysqlconnector",
    username="root",
    password="Shifa@0828", 
    host="127.0.0.1",
    database="shopsense",
)
engine = create_engine(url_object)

query ="""
WITH customer_cohort AS(
  -- CTE 1: Identify each customer's cohort month (first purchase)
  -- GROUP BY customer_unique_id ensures we get the earliest purchase date per customer
    SELECT 
        c.customer_unique_id,
        MIN(DATE_FORMAT(o.order_purchase_timestamp, '%Y%m')) as cohort_month
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status NOT IN ('cancelled', 'unavailable')
    GROUP BY c.customer_unique_id
),
customer_orders AS(
  -- CTE 2: Extract all customer orders with their purchase month
  -- We'll later compare each order month against cohort month
    SELECT 
        c.customer_unique_id,
        DATE_FORMAT(o.order_purchase_timestamp, '%Y%m') AS order_month
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status NOT IN ('cancelled', 'unavailable')
),
cohort_data AS (
  -- CTE 3: Calculate months since first purchase (month_number)
  -- month_number = 0 means first purchase month
  -- month_number = 1 means 1 month after first purchase (repeat customer), etc.
    SELECT
        co.customer_unique_id,
        cc.cohort_month,
        co.order_month,
        PERIOD_DIFF(co.order_month, cc.cohort_month) AS month_number
    FROM customer_orders co
    JOIN customer_cohort cc ON co.customer_unique_id = cc.customer_unique_id
)
-- FINAL: Aggregate cohort performance metrics
-- Count distinct customers for each month_number to get retention
SELECT
    cohort_month,
    COUNT(DISTINCT CASE WHEN month_number = 0 THEN customer_unique_id END) AS cohort_size,
    COUNT(DISTINCT CASE WHEN month_number = 1 THEN customer_unique_id END) AS month_1,
    COUNT(DISTINCT CASE WHEN month_number = 2 THEN customer_unique_id END) AS month_2,
    COUNT(DISTINCT CASE WHEN month_number = 3 THEN customer_unique_id END) AS month_3,
    COUNT(DISTINCT CASE WHEN month_number = 4 THEN customer_unique_id END) AS month_4,
    COUNT(DISTINCT CASE WHEN month_number = 5 THEN customer_unique_id END) AS month_5,
    COUNT(DISTINCT CASE WHEN month_number = 6 THEN customer_unique_id END) AS month_6
FROM cohort_data
GROUP BY cohort_month
ORDER BY cohort_month;
"""

df_cohort = pd.read_sql(query, engine)
print(df_cohort)

# Save cohort results for downstream analysis (e.g., pivot table creation)
df_cohort.to_csv('../data/processed/cohort_retention.csv', index=False)
print("\n✅ Cohort retention data saved to '../data/processed/cohort_retention.csv'")

---

## 🔄 Category Funnel Analysis

**What is it?** Funnel analysis examines product category performance across order volume, revenue, delivery reliability, and customer satisfaction.

**Key Metrics:**
- `total_orders`: Number of completed orders in category
- `total_revenue`: Sum of order values (R$ Brazilian Real)
- `avg_price`: Average item price per order
- `avg_delay_days`: Mean delivery delay (positive = late)
- `late_rate_pct`: Percentage of orders delivered late
- `avg_review_score`: Customer satisfaction (1-5 stars)

**Business Value:**
- Identifies problem categories requiring improvement (high delay rate)
- Reveals high-value vs. high-volume categories (AOV vs. order count)
- Correlates delivery performance with review scores
- Guides inventory and seller allocation decisions

**Finding:** [VALUE] product categories were analyzed; top late-delivery category had [VALUE]% delay rate

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# ──────────────────────────────────────────────────────────────────
# STEP 4: CATEGORY-LEVEL FUNNEL ANALYSIS
# ──────────────────────────────────────────────────────────────────
# Multi-table join to analyze product category performance:
# - Orders + Order Items (for order count and revenue)
# - Products + Category Translation (for category names)
# - Reviews (for customer satisfaction scores)
# - Orders + Delivery dates (for on-time delivery analysis)

url_object = URL.create(
    "mysql+mysqlconnector",
    username="root",
    password="Shifa@0828", 
    host="127.0.0.1",
    database="shopsense",
)
engine = create_engine(url_object)

query ="""
SELECT 
    -- Category name: use English translation if available, else Portuguese name
    COALESCE(t.product_category_name_english, p.product_category_name) AS category,
    COUNT(DISTINCT o.order_id) AS total_orders,
    -- Revenue: sum of all item prices (excludes discounts/coupons for this analysis)
    ROUND(SUM(oi.price), 2) AS total_revenue, 
    -- Average order value (AOV): revenue per order
    ROUND(AVG(oi.price), 2) AS avg_price,
    -- Delivery reliability: calculate delay in days
    -- Delay = estimated_delivery_date - actual_delivery_date (negative = early)
    ROUND(AVG(o.delivery_delay_days), 1) AS avg_delay_days, 
    -- Count orders delivered late (delay_days > 0)
    SUM(CASE WHEN o.delivery_delay_days > 0 THEN 1 ELSE 0 END) AS late_orders, 
    -- Late rate percentage: helps identify delivery issues
    ROUND(100.0 * (SUM(CASE WHEN o.delivery_delay_days > 0 THEN 1 ELSE 0 END)
        /COUNT(*)), 1) AS late_rate_pct,
    -- Customer satisfaction score (1-5, higher is better)
    ROUND(AVG(r.review_score), 1) AS avg_review_score
FROM orders o 
JOIN order_items oi ON o.order_id = oi.order_id 
JOIN products p ON oi.product_id = p.product_id 
-- LEFT JOIN: some products may not have English translations
LEFT JOIN product_category_name_translation t ON p.product_category_name = t.product_category_name
-- LEFT JOIN: not all orders have reviews (returns, etc.)
LEFT JOIN reviews r ON o.order_id = r.order_id
WHERE o.order_status NOT IN ('cancelled', 'unavailable')
GROUP BY category
-- Filter: only categories with meaningful sample size (100+ orders)
HAVING total_orders > 100
-- Sort: identify problem categories (high delay rate first)
ORDER BY late_rate_pct DESC
LIMIT 20;  
"""

df_funnel = pd.read_sql(query, engine)
print("\n📊 TOP 20 CATEGORIES BY DELIVERY DELAY RATE:")
print(df_funnel.to_string(index=False))

# Key insight: Placeholder for actual findings after execution
print("\n💡 Key Finding: [VALUE] categories analyzed; top late category had [VALUE]% delay rate")